In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings

warnings.filterwarnings('ignore')

# ==================== CONFIGURATION ====================
MAX_RUNTIME_HOURS = 11.0 
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600
EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_DIR = Path("results_final_resnet66")
RESULTS_DIR.mkdir(exist_ok=True)

print(f" STARTING DIAMOND PROTOCOL (CNN/RESNET-66)")
print(f" Device: {DEVICE}")
print(f" Time Budget: {MAX_RUNTIME_HOURS} Hours")

# ==================== DATASET ====================
def get_data_loaders():
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

    trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return trainloader, testloader

# ==================== MODULAR RESNET-66 ====================
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, use_bn=True, use_res=True):
        super(BasicBlock, self).__init__()
        self.use_res = use_res
        
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not use_bn)
        self.bn1 = nn.BatchNorm2d(planes) if use_bn else nn.Identity()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not use_bn)
        self.bn2 = nn.BatchNorm2d(planes) if use_bn else nn.Identity()

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            layers = [nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=not use_bn)]
            if use_bn:
                layers.append(nn.BatchNorm2d(self.expansion * planes))
            self.shortcut = nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_res:
            out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet66_Custom(nn.Module):
    def __init__(self, num_classes=10, use_bn=True, use_res=True):
        super(ResNet66_Custom, self).__init__()
        self.in_planes = 64
        self.use_bn = use_bn
        
        # CIFAR-10 Stem
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not use_bn)
        self.bn1 = nn.BatchNorm2d(64) if use_bn else nn.Identity()
        
        # ResNet-66 Architecture: [11, 11, 11, 11] blocks = 44 BasicBlocks * 2 layers + conv1 + linear = 66 layers
        self.layer1 = self._make_layer(64, 11, stride=1, use_res=use_res)
        self.layer2 = self._make_layer(128, 11, stride=2, use_res=use_res)
        self.layer3 = self._make_layer(256, 11, stride=2, use_res=use_res)
        self.layer4 = self._make_layer(512, 11, stride=2, use_res=use_res)
        
        self.linear = nn.Linear(512, num_classes)
        
        # Initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, num_blocks, stride, use_res):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(BasicBlock(self.in_planes, planes, stride, self.use_bn, use_res))
            self.in_planes = planes * BasicBlock.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

# ==================== ADVANCED METRICS LOGGER ====================
class MetricsLoggerCNN:
    def __init__(self, model, model_name):
        self.model = model
        self.model_name = model_name
        self.metrics = []
        self.activations = {}
        self.hooks = []
        self.capture_mode = False
        
        # Register hooks on Layer Groups
        layers = [model.layer1, model.layer2, model.layer3, model.layer4]
        for idx, layer in enumerate(layers):
            hook = layer.register_forward_hook(self._make_hook(f'block_{idx+1}'))
            self.hooks.append(hook)

    def _make_hook(self, name):
        def hook(module, input, output):
            if self.capture_mode:
                # GAP to get feature vector (B, C)
                pooled = F.adaptive_avg_pool2d(output, (1, 1)).flatten(1)
                self.activations[name] = pooled.detach()
        return hook

    def log_metrics(self, epoch, train_loss, val_loss, val_acc):
        metrics_dict = {
            'model': self.model_name,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_acc': val_acc
        }
        
        for name, activation in self.activations.items():
            # 1. Rank Calculation
            x = activation # Already (B, C)
            if x.shape[0] > 2000: x = x[:2000]
            
            try:
                if torch.isnan(x).any() or torch.isinf(x).any():
                    stable_rank = 1.0
                    active_pct = 0.0
                    s = torch.tensor([1.0]).to(x.device)
                else:
                    s = torch.linalg.svdvals(x.float())
                    if len(s) > 0 and s[0] > 1e-6:
                        stable_rank = ((s**2).sum() / (s[0]**2)).item()
                    else:
                        stable_rank = 1.0
                    active_pct = (activation > 0).float().mean().item()
            except:
                stable_rank = 1.0
                active_pct = 0.0
                s = torch.tensor([1.0]).to(x.device)

            metrics_dict[f'rank_{name}'] = max(1.0, stable_rank)
            metrics_dict[f'active_{name}'] = active_pct
            
            # 2. Spectrum (Deepest Layer Only: block_4)
            if 'block_4' in name and epoch == EPOCHS:
                spectrum = (s[:20] / (s[0] + 1e-8)).cpu().numpy()
                metrics_dict['spectrum'] = spectrum
        
        self.metrics.append(metrics_dict)
        self.activations.clear()
        return metrics_dict

    def get_dataframe(self):
        data = [{k: v for k, v in m.items() if k != 'spectrum'} for m in self.metrics]
        return pd.DataFrame(data)

    def get_final_spectrum(self):
        for m in reversed(self.metrics):
            if 'spectrum' in m: return m['spectrum']
        return None
    
    def cleanup(self):
        for h in self.hooks: h.remove()

# ==================== PLOTTING ====================
def save_diamond_plots_cnn(all_metrics, all_spectra):
    print("\n Generating CNN Plots...")
    try:
        # Plot A: Heatmaps
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.flatten()
        for idx, (model_name, df) in enumerate(all_metrics.items()):
            if len(df) == 0: continue
            rank_cols = sorted([c for c in df.columns if 'rank_block_' in c])
            if not rank_cols: continue
            rank_data = df[rank_cols].values.T
            sns.heatmap(rank_data, ax=axes[idx], cmap='magma', vmin=1.0, vmax=5.0)
            axes[idx].set_title(f'{model_name} Rank')
            axes[idx].set_ylabel('Block Index')
            axes[idx].set_xlabel('Epoch')
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'plot_A_heatmap.png')
        plt.close()

        # Plot B: Spectral Scree
        fig, ax = plt.subplots(figsize=(10, 6))
        for model_name, spectrum in all_spectra.items():
            if spectrum is not None:
                ax.plot(range(1, len(spectrum)+1), spectrum, marker='o', label=model_name, linewidth=2)
        ax.set_yscale('log')
        ax.set_xlabel('Singular Value Index')
        ax.set_ylabel('Normalized Singular Value (Log)')
        ax.set_title('ResNet-66 Spectral Geometry (Block 4)')
        ax.legend()
        ax.grid(True, which="both", alpha=0.3)
        plt.savefig(RESULTS_DIR / 'plot_B_scree.png')
        plt.close()

        # Plot C: Rank vs Sparsity (ResNet_Naked)
        if 'ResNet_Naked' in all_metrics:
            df = all_metrics['ResNet_Naked']
            fig, ax1 = plt.subplots(figsize=(10, 6))
            rank_cols = [c for c in df.columns if 'rank_block_' in c]
            avg_rank = df[rank_cols].mean(axis=1)
            ax1.plot(df['epoch'], avg_rank, 'b-', linewidth=3, label='Rank')
            ax1.set_xlabel('Epoch')
            ax1.set_ylabel('Average Rank', color='b')
            ax2 = ax1.twinx()
            act_cols = [c for c in df.columns if 'active_' in c]
            avg_act = df[act_cols].mean(axis=1) * 100
            ax2.plot(df['epoch'], avg_act, 'r--', linewidth=2, label='Active %')
            ax2.set_ylabel('Active Neurons (%)', color='r')
            plt.title('ResNet_Naked: Rank vs Sparsity')
            plt.savefig(RESULTS_DIR / 'plot_C_sparsity.png')
            plt.close()

        # Plot D: Final Depth Slice
        fig, ax = plt.subplots(figsize=(10, 6))
        for model_name, df in all_metrics.items():
            if len(df) == 0: continue
            last_row = df.iloc[-1]
            rank_cols = sorted([c for c in df.columns if 'rank_block_' in c])
            ranks = [last_row[c] for c in rank_cols]
            ax.plot(range(1, 5), ranks, marker='o', label=model_name)
        ax.set_xlabel('Block Index')
        ax.set_ylabel('Final Stable Rank')
        ax.set_xticks([1, 2, 3, 4])
        ax.legend()
        ax.grid(True)
        plt.savefig(RESULTS_DIR / 'plot_D_depth.png')
        plt.close()

        # Plot E: Accuracy
        fig, ax = plt.subplots(figsize=(10, 6))
        for model_name, df in all_metrics.items():
            if len(df) > 0:
                ax.plot(df['epoch'], df['val_acc'], marker='.', label=model_name)
        ax.set_ylabel('Accuracy (%)')
        ax.legend()
        ax.grid(True)
        plt.savefig(RESULTS_DIR / 'plot_E_acc.png')
        plt.close()

        print("✓ All 5 Plots Saved.")
    except Exception as e:
        print(f"Plot Error: {e}")

# ==================== MAIN ====================
def run_experiment():
    start_total = time.time()
    trainloader, testloader = get_data_loaders()
    
    configs = {
        'ResNet_Star':   {'bn': True,  'res': True},
        'ResNet_NoBN':   {'bn': False, 'res': True},
        'ResNet_NoRes':  {'bn': True,  'res': False},
        'ResNet_Naked':  {'bn': False, 'res': False},
    }
    
    all_metrics = {}
    all_spectra = {}
    
    for model_name, cfg in configs.items():
        print(f"\n MODEL: {model_name}")
        try:
            model = ResNet66_Custom(use_bn=cfg['bn'], use_res=cfg['res']).to(DEVICE)
            optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            criterion = nn.CrossEntropyLoss()
            logger = MetricsLoggerCNN(model, model_name)
            
            for epoch in range(1, EPOCHS + 1):
                if time.time() - start_total > MAX_RUNTIME_SECONDS: break
                
                # 1. TRAIN (FULL DATASET, FIXED)
                logger.capture_mode = False
                model.train()
                t_loss, count = 0, 0
                for data, target in trainloader:
                    data, target = data.to(DEVICE), target.to(DEVICE)
                    optimizer.zero_grad()
                    output = model(data)
                    
                    loss = criterion(output, target)
                    if torch.isnan(loss) or torch.isinf(loss):
                        print(f" NaN Loss at Epoch {epoch}. Skipping.")
                        loss = torch.tensor(0.0, requires_grad=True).to(DEVICE)
                    else:
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Safety
                        optimizer.step()
                    
                    t_loss += loss.item()
                    count += 1
                
                # 2. METRICS (Subset)
                model.eval()
                logger.capture_mode = True
                with torch.no_grad():
                    for i, (data, _) in enumerate(trainloader):
                        if i >= 5: break
                        data = data.to(DEVICE)
                        _ = model(data)
                logger.capture_mode = False
                
                # 3. VALIDATION
                correct, total = 0, 0
                with torch.no_grad():
                    for data, target in testloader:
                        data, target = data.to(DEVICE), target.to(DEVICE)
                        out = model(data)
                        _, p = out.max(1)
                        total += target.size(0)
                        correct += p.eq(target).sum().item()
                
                acc = 100.*correct/total
                logger.log_metrics(epoch, t_loss/count, 0.0, acc)
                print(f"  Ep {epoch}: Acc {acc:.2f}%")
                
                if epoch % 5 == 0:
                    logger.get_dataframe().to_csv(RESULTS_DIR / f'{model_name}_partial.csv', index=False)
            
            all_metrics[model_name] = logger.get_dataframe()
            all_spectra[model_name] = logger.get_final_spectrum()
            logger.get_dataframe().to_csv(RESULTS_DIR / f'{model_name}_final.csv', index=False)
            logger.cleanup()
            del model
            
        except Exception as e:
            print(f" Failed {model_name}: {e}")
            
    save_diamond_plots_cnn(all_metrics, all_spectra)
    print("CNN EXPERIMENT COMPLETE.")

if __name__ == "__main__":
    run_experiment()

 STARTING DIAMOND PROTOCOL (CNN/RESNET-66)
 Device: cuda
 Time Budget: 11.0 Hours


100%|██████████| 170M/170M [00:17<00:00, 9.48MB/s]



 MODEL: ResNet_Star
  Ep 1: Acc 33.32%
  Ep 2: Acc 40.34%
  Ep 3: Acc 53.20%
  Ep 4: Acc 60.12%
  Ep 5: Acc 66.79%
  Ep 6: Acc 70.00%
  Ep 7: Acc 71.72%
  Ep 8: Acc 73.23%
  Ep 9: Acc 77.19%
  Ep 10: Acc 77.40%
  Ep 11: Acc 81.16%
  Ep 12: Acc 81.97%
  Ep 13: Acc 80.99%
  Ep 14: Acc 83.81%
  Ep 15: Acc 85.08%
  Ep 16: Acc 85.27%
  Ep 17: Acc 86.04%
  Ep 18: Acc 86.34%
  Ep 19: Acc 87.35%
  Ep 20: Acc 88.07%

 MODEL: ResNet_NoBN
  Ep 1: Acc 10.00%
  Ep 2: Acc 10.00%
  Ep 3: Acc 10.00%
  Ep 4: Acc 10.00%
  Ep 5: Acc 10.00%
  Ep 6: Acc 10.00%
  Ep 7: Acc 10.00%
  Ep 8: Acc 10.00%
  Ep 9: Acc 10.00%
  Ep 10: Acc 10.00%
  Ep 11: Acc 10.00%
  Ep 12: Acc 10.00%
  Ep 13: Acc 10.00%
  Ep 14: Acc 10.00%
  Ep 15: Acc 10.00%
  Ep 16: Acc 10.00%
  Ep 17: Acc 10.00%
  Ep 18: Acc 10.00%
  Ep 19: Acc 10.00%
  Ep 20: Acc 10.00%

 MODEL: ResNet_NoRes
  Ep 1: Acc 13.95%
  Ep 2: Acc 10.08%
  Ep 3: Acc 9.74%
  Ep 4: Acc 15.73%
  Ep 5: Acc 19.29%
  Ep 6: Acc 14.15%
  Ep 7: Acc 12.30%
  Ep 8: Acc 21.85%
  E